# Synthetic Data — Analysis & Interpretation

Loads pre-computed results from the generation notebook (`synthetic_eval.ipynb`).

**Structure**
1. Load saved results
2. Tables
3. Figures

---
## 1. Load Saved Results

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

import ast
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import json
from plots import plot_kld_per_feature_by_method, plot_ablation_curve, plot_pca_fraction_grid, plot_kld_fraction_grid
from tables import (
    make_table_full, make_table_sample_size,
    make_table_forward, make_table_reverse,
    make_table_drop_one, save_all_tables,
)
from metrics import evaluate_abl
import matplotlib
matplotlib.use('inline')
OUTDIR = Path('../results')

df_all = pd.read_csv(OUTDIR / 'synthetic_results_all.csv')

with open(OUTDIR / 'all_figs.pkl', 'rb') as f:
    all_figs = pickle.load(f)

with open(OUTDIR / 'summary_figs.pkl', 'rb') as f:
    summary_figs = pickle.load(f)

def parse_feature_names(x):
    if not isinstance(x, str):
        return x
    return [s.strip().strip("'\"") for s in x.strip("[]").split(",") if s.strip()]

df_all["kept_feature_names"] = df_all["kept_feature_names"].apply(parse_feature_names)
df_all["kept_feature_idx"]   = df_all["kept_feature_idx"].apply(
    lambda x: [int(s.strip()) for s in x.strip("[]").split(",") if s.strip()]
    if isinstance(x, str) else x
)

print(f'Loaded {len(df_all)} rows, {len(all_figs)} figures.')
print(f'Datasets:  {df_all["dataset"].unique()}')
print(f'Methods:   {df_all["method"].unique()}')
print(f'Modes:     {df_all["feature_mode"].unique()}')

---
## 2. Tables

### Table 1 — Main Results
Full features, largest sample fraction. One row per dataset × method.

| Column | Meaning | Better |
|---|---|---|
| RF Sep ↓ | Discriminator separability (0.5 = indistinguishable) | lower |
| Disc F1 ↓ | Discriminator F1 | lower |
| TSTR F1 ↑ | Train-on-synthetic, test-on-real F1 | higher |
| Util Gap ↓ | TRTR − TSTR; cost of using synthetic instead of real | lower |
| KLD Mean ↓ | Avg KL divergence per feature | lower |
| Corr Diff ↓ | Mean absolute correlation difference | lower |

In [ ]:
largest_frac = df_all.loc[df_all['feature_mode'] == 'full', 'frac'].max()
display(make_table_full(df_all, frac=largest_frac))

### Table 2 — Sample Size Sweep
Does synthetic quality degrade as the generator sees less real data?

In [ ]:
display(make_table_sample_size(df_all))

### Table 3 — Forward Ablation
Keep top k RF-ranked features. Increasing k = more dependency complexity for the generator.

In [ ]:
for ds in df_all['dataset'].unique():
    print(f'\n{ds}')
    display(make_table_forward(df_all, dataset=ds))

### Table 4 — Reverse Ablation
Drop top k features. If RF Sep stays high, failure is distributed — not concentrated.

In [ ]:
for ds in df_all['dataset'].unique():
    print(f'\n{ds}')
    display(make_table_reverse(df_all, dataset=ds))

### Table 5 — Drop-One Sensitivity
Which single feature removal changes synthetic quality most? Sorted worst-first.

In [ ]:
for ds in df_all['dataset'].unique():
    print(f'\n{ds}')
    display(make_table_drop_one(df_all, dataset=ds))

In [ ]:
save_all_tables(df_all, outdir=OUTDIR)
print('All tables saved.')

---
## 3. Figures

### Figure helper

In [ ]:
def show_figures(figures, dataset=None, method=None, feature_mode=None, 
                 subset_label=None, frac=None, plot=None):
    """
    Display stored diagnostic figures with optional filters.
    plot: 'corr' | 'pca' | 'kld' | 'kld_array' (skip — raw array, not a figure)
    """
    for item in figures:
        if dataset      and item.get('dataset')      != dataset:      continue
        if method       and item.get('method')        != method:       continue
        if feature_mode and item.get('feature_mode')  != feature_mode: continue
        if subset_label and item.get('subset_label')  != subset_label: continue
        if frac         and item.get('frac')           != frac:         continue
        # rest unchanged

        fig_obj = item['fig']
        label   = f"{item.get('dataset')} | {item.get('method')} | {item.get('subset_label')}"

        if isinstance(fig_obj, dict):
            selected = fig_obj.items() if plot is None else \
                       [(plot, fig_obj[plot])] if plot in fig_obj else []
            for name, fig in selected:
                if name == 'kld_array': 
                    continue
                print(f'{label} | {name}')
                display(fig)
        else:
            print(label)
            display(fig_obj)

 
DATASETS_IN_RESULTS = df_all["dataset"].unique()
METHODS = ["bootstrap", "gmm", "cvae"]

### 3a. Ablation Curves — RF Sep vs. k

In [ ]:
# 3a. Ablation curves grouped by dataset
for ds in df_all["dataset"].unique():
    ds_figs = {k: v for k, v in summary_figs.items() if ds in k}
    if not ds_figs:
        continue
    print(f"\n{'='*40}\n{ds}\n{'='*40}")
    for fig in ds_figs.values():
        display(fig)
        plt.close(fig)

### 3b. PCA Projections — full features, all datasets & methods

In [ ]:
for ds in DATASETS_IN_RESULTS:
    for method in METHODS:
        fig = plot_pca_fraction_grid(all_figs, dataset=ds, method=method)
        if fig:
            print(f"\n{ds} | {method}")
            display(fig)
            plt.close(fig)
 

### 3c. Correlation Matrices — real vs synthetic vs difference

In [ ]:
from plots import plot_corr_fraction_grid
for ds in DATASETS_IN_RESULTS:
    for method in METHODS:
        fig = plot_corr_fraction_grid(all_figs, dataset=ds, method=method)
        if fig:
            print(f"\n{ds} | {method}")
            display(fig)
            plt.close(fig)
 

### 3d. KLD Per Feature — which features are hardest to synthesize?

In [ ]:
from plots import plot_kld_per_feature_by_method, plot_ablation_curve, plot_pca_fraction_grid, plot_kld_fraction_grid

for ds in DATASETS_IN_RESULTS:
    for method in METHODS:
        fig = plot_kld_fraction_grid(all_figs, dataset=ds, method=method)
        if fig:
            print(f"\n{ds} | {method}")
            display(fig)
            plt.close(fig)

In [ ]:
for ds in df_all['dataset'].unique():
    kld_dict           = {}
    feature_names_used = None

    for item in all_figs:
        if item['dataset'] != ds or item['feature_mode'] != 'full':
            continue
        if 'kld_array' not in item['fig']:
            continue

        kld_dict[item['method']] = item['fig']['kld_array']

        if feature_names_used is None:
            row = df_all[
                (df_all['dataset']      == ds) &
                (df_all['method']       == item['method']) &
                (df_all['feature_mode'] == 'full')
            ].iloc[0]
            feature_names_used = row.get('kept_feature_names', None)

    if len(kld_dict) < 2:
        print(f'{ds}: not enough methods stored — re-run generation notebook')
        continue

    print(f'\n{ds}')
    fig = plot_kld_per_feature_by_method(kld_dict, feature_names=feature_names_used, top_n=20)
    display(fig)
    plt.close(fig)